# Weather Pipeline Walkthrough

This notebook uses the same loader functions as the Airflow DAG. It loads one date at a time through the range helper, records the raw load evidence, proves rerun safety, and then builds and tests the dbt mart.

## 1. Setup

The setup imports the loader functions used by the DAG, reads the configured cities and warehouse connection, and defines a reproducible 30-day logical range from August 25 through September 23, 2026.

In [ ]:
from datetime import date, timedelta
from pathlib import Path
import subprocess

import pandas as pd
import psycopg2

from ingestion.loader import (
    get_cities,
    get_db_config,
    load_weather_for_date,
    load_weather_for_date_range,
)

logical_end_date = date(2026, 9, 23)
logical_start_date = logical_end_date - timedelta(days=29)
cities = get_cities()
db_config = get_db_config()

print(f"Cities: {len(cities)}")
print(f"Logical date range: {logical_start_date} to {logical_end_date}")
print(f"Warehouse: {db_config['host']}:{db_config['port']}/{db_config['dbname']}")

Cities: 5
Logical date range: 2026-08-25 to 2026-09-23
Warehouse: postgres:5432/warehouse


## 2. Extract and load

`load_weather_for_date_range()` is the same backfill helper used by the loader. It calls the one-date loader for each date in the range, which fetches from Open-Meteo and writes to `raw.weather_daily`. The summary and sample query provide evidence of the loaded dates, rows, and `loaded_at` timestamps.

In [2]:
def query_dataframe(sql, params=None):
    with psycopg2.connect(**db_config) as connection:
        return pd.read_sql_query(sql, connection, params=params)


load_weather_for_date_range(
    cities,
    logical_start_date.isoformat(),
    logical_end_date.isoformat(),
    db_config,
)

raw_summary = query_dataframe(
    """
    SELECT MIN(date) AS first_date,
           MAX(date) AS last_date,
           COUNT(DISTINCT date) AS loaded_days,
           COUNT(*) AS row_count,
           COUNT(loaded_at) AS timestamped_rows
    FROM raw.weather_daily
    """
)
raw_samples = query_dataframe(
    """
    SELECT city, date, temperature_2m_mean, temperature_2m_min,
           temperature_2m_max, precipitation_sum, loaded_at
    FROM raw.weather_daily
    ORDER BY date, city
    LIMIT 10
    """
)

print("Raw load summary:")
display(raw_summary)
print("Raw sample rows:")
display(raw_samples)

Raw load summary:


/tmp/ipykernel_382/3872349608.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(sql, connection, params=params)


,first_date,last_date,loaded_days,row_count,timestamped_rows
0,2026-08-25,2026-09-23,30,150,150


Raw sample rows:


,city,date,temperature_2m_mean,temperature_2m_min,temperature_2m_max,precipitation_sum,loaded_at
0,Bengaluru,2026-08-25,22.8,20.1,27.7,6.3,2026-09-23 11:14:14.995139+00:00
1,Chennai,2026-08-25,29.3,26.5,32.8,2.3,2026-09-23 11:14:14.995139+00:00
2,Mumbai,2026-08-25,27.7,25.9,29.4,6.3,2026-09-23 11:14:14.995139+00:00
3,Seoul,2026-08-25,27.0,24.6,31.2,3.5,2026-09-23 11:14:14.995139+00:00
4,Tokyo,2026-08-25,30.9,27.7,34.9,0.0,2026-09-23 11:14:14.995139+00:00
5,Bengaluru,2026-08-26,23.6,20.2,28.7,1.9,2026-09-23 11:14:22.219898+00:00
6,Chennai,2026-08-26,30.6,28.0,34.6,1.6,2026-09-23 11:14:22.219898+00:00
7,Mumbai,2026-08-26,27.5,26.1,29.4,6.2,2026-09-23 11:14:22.219898+00:00
8,Seoul,2026-08-26,26.9,24.5,29.7,0.3,2026-09-23 11:14:22.219898+00:00
9,Tokyo,2026-08-26,31.2,26.3,35.9,0.1,2026-09-23 11:14:22.219898+00:00


## 3. Rerun safety

The final logical date is loaded twice with `load_weather_for_date()`. The helper deletes existing rows for that date before inserting fresh rows, so the parameterized count query should show the same count before and after both loads.

In [ ]:
def count_rows_for_date(logical_date):
    query = "SELECT COUNT(*) AS row_count FROM raw.weather_daily WHERE date = %s"
    return int(query_dataframe(query, params=(logical_date,)).iloc[0]["row_count"])

rerun_date = logical_end_date.isoformat()

count_before = count_rows_for_date(rerun_date)

# First Rerun
load_weather_for_date(cities, rerun_date, db_config)
count_after_first_load = count_rows_for_date(rerun_date)

# Second Rerun
load_weather_for_date(cities, rerun_date, db_config)
count_after_second_load = count_rows_for_date(rerun_date)

print(f"Rows before rerun: {count_before}")
print(f"Rows after first load: {count_after_first_load}")
print(f"Rows after second load: {count_after_second_load}")

assert count_before == count_after_first_load == count_after_second_load
print("Rerun safety check passed: the row count did not change.")

/tmp/ipykernel_382/3872349608.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(sql, connection, params=params)


Rows before rerun: 5
Rows after first load: 5
Rows after second load: 5
Rerun safety check passed: the row count did not change.


## 4. dbt run

This cell executes the same command as the DAG: `cd /opt/airflow/dbt/weather_dbt && dbt run`. It rebuilds the raw-schema staging view and daily fact table from the loaded source data.

In [4]:
dbt_directory = Path("/opt/airflow/dbt/weather_dbt")
run_result = subprocess.run(
    ["bash", "-lc", "cd /opt/airflow/dbt/weather_dbt && dbt run"],
    capture_output=True,
    text=True,
    check=False,
)
print(run_result.stdout)
print(run_result.stderr)
assert run_result.returncode == 0, run_result.stdout + run_result.stderr

11:17:33  Running with dbt=1.8.8
11:17:33  Registered adapter: postgres=1.8.2
11:17:33  Found 2 models, 12 data tests, 1 source, 423 macros
11:17:33  
11:17:34  Concurrency: 4 threads (target='dev')
11:17:34  
11:17:34  1 of 2 START sql view model raw.stg_weather .................................... [RUN]
11:17:34  1 of 2 OK created sql view model raw.stg_weather ............................... [CREATE VIEW in 0.33s]
11:17:34  2 of 2 START sql table model raw.fct_city_daily ................................ [RUN]
11:17:34  2 of 2 OK created sql table model raw.fct_city_daily ........................... [SELECT 150 in 0.33s]
11:17:34  
11:17:34  Finished running 1 view model, 1 table model in 0 hours 0 minutes and 0.92 seconds (0.92s).
11:17:35  
11:17:35  Completed successfully
11:17:35  
11:17:35  Done. PASS=2 WARN=0 ERROR=0 SKIP=0 TOTAL=2




## 5. dbt test

This cell executes the same `dbt test` command as the DAG after the models are built. The tests check uniqueness, nulls, temperature ranges, precipitation values, and source coordinate ranges.

In [5]:
test_result = subprocess.run(
    ["bash", "-lc", "cd /opt/airflow/dbt/weather_dbt && dbt test"],
    capture_output=True,
    text=True,
    check=False,
)
print(test_result.stdout)
print(test_result.stderr)
assert test_result.returncode == 0, test_result.stdout + test_result.stderr

11:17:39  Running with dbt=1.8.8
11:17:39  Registered adapter: postgres=1.8.2
11:17:40  Found 2 models, 12 data tests, 1 source, 423 macros
11:17:40  
11:17:40  Concurrency: 4 threads (target='dev')
11:17:40  
11:17:41  1 of 12 START test fct_city_daily_unique_city_date ............................. [RUN]
11:17:41  2 of 12 START test max_temperature_not_less_than_min ........................... [RUN]
11:17:41  3 of 12 START test not_null_fct_city_daily_temperature_range ................... [RUN]
11:17:41  4 of 12 START test not_null_stg_weather_city ................................... [RUN]
11:17:41  1 of 12 PASS fct_city_daily_unique_city_date ................................... [PASS in 0.19s]
11:17:41  2 of 12 PASS max_temperature_not_less_than_min ................................. [PASS in 0.20s]
11:17:41  5 of 12 START test not_null_stg_weather_date ................................... [RUN]
11:17:41  3 of 12 PASS not_null_fct_city_daily_temperature_range ......................... 

## 6. Business result

The final query reads `raw.fct_city_daily`, the dbt mart in this project. It returns the latest date for each city with mean, minimum, maximum, temperature range and precipitation values that a business user can compare.

In [6]:
latest_city_weather = query_dataframe(
    """
    SELECT city,
           date,
           mean_temperature,
           min_temperature,
           max_temperature,
           temperature_range,
           total_precipitation
    FROM raw.fct_city_daily
    WHERE date = (SELECT MAX(date) FROM raw.fct_city_daily)
    ORDER BY city
    """
)
display(latest_city_weather)

/tmp/ipykernel_382/3872349608.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(sql, connection, params=params)


,city,date,mean_temperature,min_temperature,max_temperature,temperature_range,total_precipitation
0,Bengaluru,2026-09-23,22.7,19.6,27.4,7.8,4.5
1,Chennai,2026-09-23,28.3,26.5,31.6,5.1,4.5
2,Mumbai,2026-09-23,27.4,24.9,30.4,5.5,1.1
3,Seoul,2026-09-23,23.3,19.2,28.9,9.7,0.1
4,Tokyo,2026-09-23,20.3,19.4,22.4,3.0,12.0
